# Step 1 — Source Separation (Path A)

Isolate the **piano stem** from a mixed jazz recording using `audio-separator`.

**Default model:** BS-RoFormer 6-stem (best piano stem quality in 2025/2026).
**Fallback:** `htdemucs_6s` (faster, weaker piano).

Runs on:
- **Google Colab** (free T4 GPU): ~30–90 s for a 3-minute song.
- **CPU (your laptop)**: ~5–10 min per song. Pick `htdemucs_6s` if patient is not your style.

## How to use
1. Drop your audio file into the `input/` folder of the repo (e.g. `input/coltrane.mp3`).
2. Run all cells. The piano stem lands in `output/`.
3. Listen to the preview cell at the bottom to sanity-check the separation.

## 0. Environment setup

**Colab:** uncomment the `pip install` cell. **Local (.venv):** install once via `pip install -r requirements.txt` from your terminal — no need to re-run inside the notebook.

In [1]:
# Colab only — uncomment if running in Colab
# !pip install -q "audio-separator[gpu]>=0.28.0" soundfile tqdm

# If running locally inside an activated .venv, you should already have these.
import sys
print('Python:', sys.version)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [ ]:
from pathlib import Path

# If the notebook is opened from notebooks/, the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_DIR = PROJECT_ROOT / 'input'
OUTPUT_DIR = PROJECT_ROOT / 'output'
MODEL_DIR = PROJECT_ROOT / 'models'

INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Input  files :', sorted(p.name for p in INPUT_DIR.iterdir() if p.is_file() and not p.name.startswith('.')))

## 1. Pick the input file and the model

In [ ]:
# ---- Edit these two variables ----
INPUT_FILENAME = 'song.mp3'   # e.g. 'coltrane_giant_steps.mp3' (must live inside input/)
MODEL_KEY      = 'bs_roformer'  # 'bs_roformer' (recommended) or 'htdemucs_6s' (faster, weaker piano)
# ----------------------------------

MODELS = {
    'bs_roformer': 'model_bs_roformer_ep_368_sdr_12.9628.ckpt',
    'htdemucs_6s': 'htdemucs_6s.yaml',
}

input_path = INPUT_DIR / INPUT_FILENAME
assert input_path.exists(), f'Drop your audio file at {input_path} before running this cell.'
model_filename = MODELS[MODEL_KEY]
print(f'Input : {input_path}')
print(f'Model : {model_filename}')

## 2. Load the separator and run

First run downloads the checkpoint to `models/` (~400 MB for BS-RoFormer). Subsequent runs reuse it.

In [ ]:
from audio_separator.separator import Separator
import time

separator = Separator(
    output_dir=str(OUTPUT_DIR),
    model_file_dir=str(MODEL_DIR),
    output_format='WAV',
)
separator.load_model(model_filename=model_filename)

t0 = time.time()
output_files = separator.separate(str(input_path))
elapsed = time.time() - t0
print(f'Separation took {elapsed:.1f}s')
print('Output files:')
for f in output_files:
    print('  -', f)

## 3. Locate the piano stem

In [ ]:
output_paths = [OUTPUT_DIR / f for f in output_files]
piano_stem = next((p for p in output_paths if 'piano' in p.name.lower()), None)

if piano_stem is None:
    print('No piano stem detected. Available stems:')
    for p in output_paths:
        print('  -', p.name)
else:
    print('Piano stem :', piano_stem)

## 4. Listen to the result

Sanity-check by ear before piping into Step 2 (AMT).

In [ ]:
from IPython.display import Audio, display

if piano_stem is not None and piano_stem.exists():
    print('Piano stem:')
    display(Audio(str(piano_stem)))
else:
    print('Piano stem not available.')

print('Original mix:')
display(Audio(str(input_path)))

## Sanity-check tips

- **Solo-piano test file**: separation should produce near-silence on every stem except piano. If bass/drums leak loudly, the model is mis-fitting.
- **Jazz combo file**: piano should be intelligible, with minimal drum bleed and no obvious bass leakage. Some reverb tail loss is normal.
- If quality is poor, try `MODEL_KEY = 'htdemucs_6s'` and compare.